# casmi26 align speed trial (Issue #99)
Speed only (not used for evaluation): frozen DreaMS `ssl_model.ckpt` (Zenodo 10997887, md5 checked) forward on 10,000 fold-safe train spectra and one epoch of a provisional alignment MLP on T4. Writes `align_speed_report.json` only.


In [ ]:
ALIGN_SRC = '#!/usr/bin/env python3\n"""整列チャネル（凍結 DreaMS ssl_model.ckpt ＋ 整列 MLP）の速度を Kaggle T4 で 1 回測る（Issue #99、計画 2026-11-14 昼 項目 5）。\n\n学習の本番ではない。#99 の条件 (2)「1 万スペクトルの速度を測り、外挿が 6 時間を超えるなら止める」の材料だけを作る。\n事前登録: https://github.com/taichiiiiiiii/molecule-id-from-mass-spectra/issues/99#issuecomment-6870838727\n\nノートの中では、このファイルを /kaggle/tmp_align に書いて import し、`run()` を呼ぶ。\n段の順: fp32 の段（ピーク 60・100 の前向き、MLP）→ 外挿と判定 → fp16 の参考の段。fp16 の段は失敗しても記録して続ける\n（DreaMS の注意の masked_fill(-1e9) は fp16 であふれうるため）。判定は fp32 だけで決まる。\n出力は align_speed_report.json だけ。どの段で止まっても report を書いて終える（数値は作らない）。\n\ntorch はモジュールの先頭で import しない（CI は torch を入れないので、純粋な関数だけをテストする）。\n"""\nfrom __future__ import annotations\n\nimport hashlib\nimport importlib.abc\nimport importlib.machinery\nimport json\nimport math\nimport os\nimport resource\nimport subprocess\nimport sys\nimport time\nimport types\nimport urllib.request\nimport zipfile\nfrom pathlib import Path\n\nZENODO_URL = "https://zenodo.org/api/records/10997887/files/ssl_model.ckpt/content"\nCKPT_SIZE = 1_392_710_212\nCKPT_MD5 = "4deee91dd4486c2b6cbcde1f200b8f50"  # Zenodo API files[].checksum（2026-11-14 12:13 JST に取得）\nDREAMS_COMMIT = "dbec3a0b514a99e5056cfccde4559fda8cfe8129"\nDREAMS_ZIP_URL = f"https://github.com/pluskal-lab/DreaMS/archive/{DREAMS_COMMIT}.zip"\n\nN_SPECTRA = 10_000\nEMB_BATCH = 32              # DreaMS API（dreams_intermediates / dreams_embeddings）の既定\nPEAKS = (60, 100)           # 60 = ckpt の max_peaks_n・dreams_intermediates の既定、100 = PreTrainedModel.from_name の既定\nTRAIN_COLS = ["inchikey14", "ms2_mzs", "ms2_normalized_intensities", "precursor_mz", "collision_energy_ev", "adduct"]\nTRAIN_ROWS = 2_539_608\nREAD_BATCH = 8192          # parquet を読む単位（pyarrow iter_batches）\n\n# 整列 MLP（#99 の文書 (e) の形で仮に置いたもの。条件 (1) の前なので本番の代表値にならない可能性がある）\nMLP_BATCH = 128\nMLP_K_NEG = 128\nMLP_STEPS_PROD = 30_000     # massspecgym_formula.yaml の n_max_steps\nMLP_WARMUP_EXCLUDED = 3\nMLP_LR = 1e-4\nMLP_WD = 1e-6\nMLP_CLIP = 5.0\nMLP_TEMP_INIT = 0.07\nMLP_DROPOUT = 0.30\nFP_BITS = 10_226\nD_OUT = 512\nMOL_HIDDEN = 2048\nMOL_LAYERS = 3\nMS_HIDDEN = 512\nMS_POST_LAYERS = 2\nMETA_DIM = 64\nFP_DENSITY = 0.03           # 合成の指紋の 1 の割合（密な行列積なので計算量には効かない）\nFP_BANK = 20_000\n\n# 外挿の前提（#99 の数字）\nPROD_TRAIN_SPECTRA = 1_982_405\nPROD_TRAIN_STRUCTS = 220_443\nPROD_QUERY_SPECTRA = 4_756\nLIMIT_HOURS = 6.0\n\nBUDGET_S = 1800             # ノート全体の打ち切り\nSTAGE_CUTOFF_S = 1500       # これを過ぎたら新しい段を始めない\nDOWNLOAD_TIMEOUT_S = 720\n\n# 前向きに要らない外部パッケージ。import の前に stub を入れる（すでに import 済みのものは本物を使う）\nSTUB_TOP = ("pytorch_lightning", "torchmetrics", "matchms", "pandarallel", "igraph", "umap", "pynndescent",\n            "plotly", "h5py", "rdkit", "numba", "networkx", "sklearn", "scipy", "seaborn", "ase", "wandb",\n            "fire", "huggingface_hub", "matplotlib", "tqdm", "pyopenms", "molplotly", "pylab", "pyteomics", "lxml",\n            "click")\n\n\nclass Md5Mismatch(RuntimeError):\n    pass\n\n\nclass StageTimeout(RuntimeError):\n    pass\n\n\n# ---------------------------------------------------------------- 純粋な関数（CI でテストする）\n\ndef md5_stream(chunks, expected_md5: str, expected_size: int | None = None) -> dict:\n    """バイト列の反復から md5 とサイズを計算し、完全な値で照合する。違えば Md5Mismatch。"""\n    h, n = hashlib.md5(), 0\n    for c in chunks:\n        h.update(c)\n        n += len(c)\n    got = h.hexdigest()\n    info = {"expected_md5": expected_md5, "md5": got, "size": n, "expected_size": expected_size}\n    if len(expected_md5) != 32:\n        raise ValueError(f"expected md5 must be the full 32-hex value, got {expected_md5!r}")\n    if got != expected_md5 or (expected_size is not None and n != expected_size):\n        raise Md5Mismatch(json.dumps(info))\n    return info\n\n\ndef verify_md5_file(path, expected_md5: str, expected_size: int | None = None) -> dict:\n    def it():\n        with open(path, "rb") as f:\n            while True:\n                b = f.read(1 << 22)\n                if not b:\n                    return\n                yield b\n    return md5_stream(it(), expected_md5, expected_size)\n\n\ndef extrapolate(sps_by_peaks: dict, mlp_steps_per_s: float, read_s: float, rows_read: int,\n                fixed_s: float) -> dict:\n    """事前登録の式。sps_by_peaks = {ピーク数: fp32 の（前処理＋前向き）スペクトル/秒}。"""\n    n_embed = PROD_TRAIN_SPECTRA + PROD_QUERY_SPECTRA\n    worst_peaks = min(sps_by_peaks, key=lambda k: sps_by_peaks[k])\n    sps = sps_by_peaks[worst_peaks]\n    t_read = read_s * (TRAIN_ROWS / rows_read)\n    t_embed = n_embed / sps\n    t_mlp = MLP_STEPS_PROD / mlp_steps_per_s\n    total = t_read + t_embed + t_mlp + fixed_s\n    hours = total / 3600.0\n    return {\n        "formula": "T_total = T_read + T_embed + T_mlp + T_fixed; T_read = read_s*(2539608/rows_read); "\n                   "T_embed = (1982405+4756)/min_peaks(sps_fp32); T_mlp = 30000/steps_per_s_fp32; "\n                   "T_fixed = download+code+load seconds of this run",\n        "premises": {"train_spectra": PROD_TRAIN_SPECTRA, "train_structures": PROD_TRAIN_STRUCTS,\n                     "query_spectra": PROD_QUERY_SPECTRA, "mlp_steps": MLP_STEPS_PROD, "train_rows": TRAIN_ROWS,\n                     "mlp_config": "provisional (#99 doc (e) shape, before condition (1))",\n                     "precision": "fp32 (fp16 autocast is reference only)"},\n        "worst_peaks": worst_peaks, "sps_used": sps,\n        "T_read_s": t_read, "T_embed_s": t_embed, "T_mlp_s": t_mlp, "T_fixed_s": fixed_s,\n        "T_total_s": total, "T_total_hours": hours, "limit_hours": LIMIT_HOURS,\n        "decision": "candidate_for_next_plan" if hours <= LIMIT_HOURS else "stop_production",\n    }\n\n\nclass _StubMeta(type):\n    """何を取り出しても stub のクラスを返す。呼ぶと（デコレータとして関数 1 つなら）その関数、それ以外は自分を返す。"""\n\n    def __getattr__(cls, name):\n        if name.startswith("__") and name.endswith("__"):\n            raise AttributeError(name)\n        return _stub_class(f"{cls.__name__}.{name}")\n\n    def __call__(cls, *a, **k):\n        if len(a) == 1 and not k and callable(a[0]) and not isinstance(a[0], _StubMeta):\n            return a[0]\n        return cls\n\n    def __getitem__(cls, k):\n        return cls\n\n    def __setitem__(cls, k, v):\n        pass\n\n    def __iter__(cls):\n        return iter(())\n\n    def __or__(cls, other):\n        return cls\n\n    __ror__ = __or__\n\n\ndef _stub_class(name: str):\n    return _StubMeta(name, (), {})\n\n\nclass _StubModule(types.ModuleType):\n    def __getattr__(self, name):\n        if name.startswith("__") and name.endswith("__"):\n            raise AttributeError(name)\n        v = _stub_class(f"{self.__name__}.{name}")\n        setattr(self, name, v)\n        return v\n\n\ndef _lightning_stub(name: str):\n    import torch\n\n    class LightningModule(torch.nn.Module):\n        def save_hyperparameters(self, *a, **k):\n            pass\n\n        @property\n        def device(self):  # pytorch_lightning の LightningModule.device と同じ意味（DreaMS の forward が読む）\n            return next(self.parameters()).device\n\n        @property\n        def dtype(self):\n            return next(self.parameters()).dtype\n\n    m = _StubModule(name)\n    m.LightningModule = LightningModule\n    return m\n\n\nclass StubFinder(importlib.abc.MetaPathFinder, importlib.abc.Loader):\n    def __init__(self, tops):\n        self.tops = set(tops)\n        self.made = set()\n\n    def find_spec(self, fullname, path=None, target=None):\n        if fullname.split(".")[0] in self.tops:\n            return importlib.machinery.ModuleSpec(fullname, self, is_package=True)\n        return None\n\n    def create_module(self, spec):\n        self.made.add(spec.name)\n        if spec.name == "pytorch_lightning":\n            m = _lightning_stub(spec.name)\n        else:\n            m = _StubModule(spec.name)\n        m.__path__ = []\n        return m\n\n    def exec_module(self, module):\n        pass\n\n\ndef install_stubs(tops=STUB_TOP) -> StubFinder:\n    active = [t for t in tops if t not in sys.modules]\n    f = StubFinder(active)\n    sys.meta_path.insert(0, f)\n    return f\n\n\n# ---------------------------------------------------------------- Kaggle で動く段\n\ndef _now(t0):\n    return round(time.time() - t0, 3)\n\n\ndef ram_peak() -> dict:\n    out = {"ru_maxrss_bytes": resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * (1 if sys.platform == "darwin" else 1024)}\n    for p in ("/sys/fs/cgroup/memory.peak", "/sys/fs/cgroup/memory/memory.max_usage_in_bytes"):\n        try:\n            out[p] = int(Path(p).read_text().strip())\n        except Exception:\n            pass\n    return out\n\n\ndef download_ckpt(dest: Path, url=ZENODO_URL, expected_md5=CKPT_MD5, expected_size=CKPT_SIZE,\n                  timeout_s=DOWNLOAD_TIMEOUT_S) -> dict:\n    t0 = time.time()\n\n    got = [0]\n\n    def chunks():\n        with urllib.request.urlopen(url, timeout=60) as r, open(dest, "wb") as f:\n            while True:\n                b = r.read(1 << 22)\n                if not b:\n                    return\n                if time.time() - t0 > timeout_s:\n                    raise StageTimeout(f"download exceeded {timeout_s}s after {got[0]} bytes "\n                                       f"({got[0] / max(time.time() - t0, 1e-9) / 1e6:.2f} MB/s)")\n                f.write(b)\n                got[0] += len(b)\n                yield b\n    info = md5_stream(chunks(), expected_md5, expected_size)\n    info["seconds"] = round(time.time() - t0, 3)\n    info["mb_per_s"] = info["size"] / max(info["seconds"], 1e-9) / 1e6\n    return info\n\n\ndef fetch_dreams_code(dest: Path, url=DREAMS_ZIP_URL, commit=DREAMS_COMMIT) -> dict:\n    t0 = time.time()\n    zpath = dest / "dreams.zip"\n    with urllib.request.urlopen(url, timeout=120) as r:\n        zpath.write_bytes(r.read())\n    sha = hashlib.sha256(zpath.read_bytes()).hexdigest()\n    with zipfile.ZipFile(zpath) as z:\n        tops = {n.split("/")[0] for n in z.namelist()}\n        assert tops == {f"DreaMS-{commit}"}, tops\n        z.extractall(dest)\n    root = dest / f"DreaMS-{commit}"\n    assert (root / "dreams/models/dreams/dreams.py").is_file()\n    return {"root": str(root), "zip_sha256": sha, "seconds": round(time.time() - t0, 3)}\n\n\ndef load_dreams(dreams_root: str, ckpt, device: str = "cpu", stub_tops=STUB_TOP):\n    """DreaMS（commit 固定）のモデルを作り、ssl_model.ckpt の重みを strict に読む。ckpt はパスか file-like。"""\n    import torch  # noqa: F401\n    if dreams_root not in sys.path:\n        sys.path.insert(0, dreams_root)\n    finder = install_stubs(stub_tops)\n    try:\n        import dreams.utils.dformats as dfm\n        import dreams.utils.data as ddata\n        from dreams.models.dreams.dreams import DreaMS\n        msml = types.ModuleType("msml"); msml.__path__ = []\n        msml_utils = types.ModuleType("msml.utils"); msml_utils.__path__ = []\n        sys.modules.setdefault("msml", msml)\n        sys.modules.setdefault("msml.utils", msml_utils)\n        sys.modules.setdefault("msml.utils.dformats", dfm)\n        sys.modules.setdefault("msml.utils.data", ddata)\n        state = torch.load(ckpt, map_location="cpu", weights_only=False)\n        hp = state["hyper_parameters"]\n        model = DreaMS(hp["args"], hp["spec_preproc"])\n        missing = model.load_state_dict(state["state_dict"], strict=True)\n        for name in ("ff_out", "mz_masking_loss", "ro_out"):   # PreTrainedModel.remove_unused_backbone_parameters\n            if hasattr(model, name):\n                delattr(model, name)\n        model = model.to(device).eval()\n        info = {"stubbed_modules": sorted(finder.made), "n_layers": model.n_layers, "d_model": model.d_model,\n                "params_backbone": int(sum(p.numel() for p in model.parameters())),\n                "ckpt_global_step": state.get("global_step"), "ckpt_pl_version": state.get("pytorch-lightning_version"),\n                "load_state_dict": str(missing)}\n        del state\n        return model, ddata.SpectrumPreprocessor, dfm.DataFormatA, info\n    finally:\n        if finder in sys.meta_path:\n            sys.meta_path.remove(finder)\n        for name in list(finder.made):   # 後で本物を import できるように stub を sys.modules から外す\n            sys.modules.pop(name, None)\n\n\ndef find_inputs(root=Path("/kaggle/input")):\n    trains = [p for p in sorted(root.rglob("train.parquet")) if "casmi26" in str(p).lower()]\n    splits = [p for p in sorted(root.rglob("split.parquet")) if "fold-safe-fpnet" in str(p)]\n    return (trains[0] if trains else None), (splits[0] if splits else None)\n\n\ndef select_spectra(train_path, split_path, n=N_SPECTRA) -> tuple[list, dict]:\n    import pyarrow.parquet as pq\n    t0 = time.time()\n    drop, source, empty = set(), "fallback_head", 0\n    if split_path is not None:\n        try:\n            drop = set(pq.read_table(split_path, columns=["inchikey14"]).column("inchikey14").to_pylist())\n            source = "foldsafe_split_excluded"\n        except Exception as e:  # split が読めなければ先頭から\n            source = f"fallback_head ({type(e).__name__})"\n            drop = set()\n    t_split = time.time() - t0\n    pf = pq.ParquetFile(train_path)\n    out, rows_read = [], 0   # rows_read = parquet から読んで展開した行数（外挿の T_read の分母）\n    for batch in pf.iter_batches(batch_size=READ_BATCH, columns=TRAIN_COLS):\n        t = batch.to_pydict()\n        m = len(t["inchikey14"])\n        rows_read += m\n        for i in range(m):\n            if t["inchikey14"][i] in drop:\n                continue\n            mz, it = t["ms2_mzs"][i], t["ms2_normalized_intensities"][i]\n            if not mz:\n                empty += 1\n                continue\n            ce = t["collision_energy_ev"][i]\n            out.append((mz, it, float(t["precursor_mz"][i]), (ce[0] if ce else None), t["adduct"][i]))\n            if len(out) >= n:\n                break\n        if len(out) >= n:\n            break\n    info = {"data_source": source, "n_drop_structures": len(drop), "rows_read": rows_read, "n_selected": len(out),\n            "n_skipped_empty_spectra": empty, "read_batch": READ_BATCH, "seconds_split_read": round(t_split, 3),\n            "seconds": round(time.time() - t0, 3)}   # seconds = split の読み＋parquet の読み＋選別（外挿の T_read に使う）\n    return out, info\n\n\ndef preprocess(specs, SpectrumPreprocessor, DataFormatA, n_peaks):\n    import numpy as np\n    pp = SpectrumPreprocessor(dformat=DataFormatA(), n_highest_peaks=n_peaks)\n    t0 = time.time()\n    arr = np.stack([pp(np.array([mz, it], dtype=np.float64), prec_mz=prec, high_form=False)\n                    for mz, it, prec, _, _ in specs]).astype(np.float32)\n    return arr, time.time() - t0\n\n\ndef embed_speed(model, arr, device, amp: bool, batch=EMB_BATCH, deadline=None):\n    import torch\n    x = torch.from_numpy(arr)\n    out = torch.zeros((len(arr), model.d_model), dtype=torch.float32)\n    store = {}\n    last = model.transformer_encoder.ffs[model.n_layers - 1]\n    h = last.register_forward_hook(lambda mod, inp, o: store.__setitem__("e", o[:, 0, :]))\n    torch.cuda.reset_peak_memory_stats() if device == "cuda" else None\n    try:\n        with torch.inference_mode():   # 2 バッチの空回し（計測から除く）\n            for i in range(0, min(2 * batch, len(arr)), batch):\n                with torch.autocast("cuda", dtype=torch.float16, enabled=amp and device == "cuda"):\n                    model(x[i:i + batch].to(device))\n        if device == "cuda":\n            torch.cuda.synchronize()\n        t0, done = time.time(), 0\n        with torch.inference_mode():\n            for i in range(0, len(arr), batch):\n                if deadline is not None and time.time() > deadline:\n                    raise StageTimeout(f"embedding stopped at {done} spectra")\n                with torch.autocast("cuda", dtype=torch.float16, enabled=amp and device == "cuda"):\n                    model(x[i:i + batch].to(device, non_blocking=True))\n                out[i:i + batch] = store["e"].float().cpu()\n                done = min(i + batch, len(arr))\n        if device == "cuda":\n            torch.cuda.synchronize()\n        dt = time.time() - t0\n    finally:\n        h.remove()\n    mem = ({"max_allocated": torch.cuda.max_memory_allocated(), "max_reserved": torch.cuda.max_memory_reserved()}\n           if device == "cuda" else {})\n    return out, {"seconds_forward": dt, "spectra": len(arr), "forward_sps": len(arr) / dt, "gpu_mem": mem,\n                 "finite": bool(torch.isfinite(out).all())}\n\n\ndef build_mlps(device):\n    import torch\n    from torch import nn\n\n    def mlp(d_in, d_hidden, n_hidden, d_out):\n        layers, d = [], d_in\n        for _ in range(n_hidden):\n            layers += [nn.Linear(d, d_hidden), nn.ReLU(), nn.Dropout(MLP_DROPOUT)]\n            d = d_hidden\n        layers.append(nn.Linear(d, d_out))\n        return nn.Sequential(*layers)\n\n    class Align(nn.Module):\n        def __init__(self, n_adducts):\n            super().__init__()\n            self.adduct = nn.Embedding(n_adducts, META_DIM // 2)\n            self.ce = nn.Linear(1, META_DIM // 2)\n            self.g = mlp(1024 + META_DIM, MS_HIDDEN, MS_POST_LAYERS, D_OUT)\n            self.h = mlp(FP_BITS, MOL_HIDDEN, MOL_LAYERS, D_OUT)\n            self.logit_scale = nn.Parameter(torch.tensor(math.log(1.0 / MLP_TEMP_INIT)))\n\n        def forward(self, emb, adduct_id, ce, fps):\n            meta = torch.cat([self.adduct(adduct_id), self.ce(ce.unsqueeze(-1))], dim=-1)\n            g = nn.functional.normalize(self.g(torch.cat([emb, meta], dim=-1)), dim=-1)\n            b = emb.shape[0]\n            hh = nn.functional.normalize(self.h(fps), dim=-1).view(b, -1, D_OUT)\n            return (g.unsqueeze(1) * hh).sum(-1) * self.logit_scale.exp()\n    return Align\n\n\ndef mlp_speed(emb, specs, device, amp: bool, n_steps=None, bank_size=FP_BANK, deadline=None, seed=0):\n    import torch\n    from torch import nn\n    g = torch.Generator(device="cpu").manual_seed(seed)\n    n = emb.shape[0]\n    steps_epoch = math.ceil(n / MLP_BATCH)\n    n_steps = steps_epoch if n_steps is None else n_steps\n    adducts = sorted({str(s[4]) for s in specs})\n    aid = torch.tensor([adducts.index(str(s[4])) for s in specs])\n    ce = torch.tensor([float(s[3]) if s[3] is not None else 0.0 for s in specs]) / 100.0\n    if device == "cuda":\n        torch.cuda.reset_peak_memory_stats()\n    bank = (torch.rand((bank_size, FP_BITS), generator=g) < FP_DENSITY).float().to(device)\n    pos = torch.randint(0, bank_size, (n,), generator=g)\n    model = build_mlps(device)(len(adducts)).to(device).train()\n    opt = torch.optim.AdamW(model.parameters(), lr=MLP_LR, weight_decay=MLP_WD)\n    _gs = getattr(getattr(torch, "amp", None), "GradScaler", None)\n    scaler = (_gs("cuda", enabled=amp and device == "cuda") if _gs is not None\n              else torch.cuda.amp.GradScaler(enabled=amp and device == "cuda"))\n    emb_d, aid_d, ce_d = emb.to(device), aid.to(device), ce.to(device)\n    order = torch.randperm(n, generator=g)\n    losses, t_start = [], None\n    total = n_steps + MLP_WARMUP_EXCLUDED\n    for step in range(total):\n        if step == MLP_WARMUP_EXCLUDED:\n            if device == "cuda":\n                torch.cuda.synchronize()\n            t_start = time.time()\n        if deadline is not None and time.time() > deadline:\n            raise StageTimeout(f"mlp stopped at step {step}")\n        j = step % steps_epoch\n        idx = order[j * MLP_BATCH:(j + 1) * MLP_BATCH]\n        b = len(idx)\n        neg = torch.randint(0, bank_size, (b, MLP_K_NEG), generator=g)\n        cand = torch.cat([pos[idx].unsqueeze(1), neg], dim=1).reshape(-1).to(device)\n        fps = bank[cand]\n        idx_d = idx.to(device)\n        with torch.autocast("cuda", dtype=torch.float16, enabled=amp and device == "cuda"):\n            logits = model(emb_d[idx_d], aid_d[idx_d], ce_d[idx_d], fps)\n            loss = nn.functional.cross_entropy(logits.float(), torch.zeros(b, dtype=torch.long, device=device))\n        opt.zero_grad(set_to_none=True)\n        scaler.scale(loss).backward()\n        scaler.unscale_(opt)\n        torch.nn.utils.clip_grad_norm_(model.parameters(), MLP_CLIP)\n        scaler.step(opt)\n        scaler.update()\n        losses.append(float(loss.detach()))\n    if device == "cuda":\n        torch.cuda.synchronize()\n    dt = time.time() - t_start\n    mem = ({"max_allocated": torch.cuda.max_memory_allocated(), "max_reserved": torch.cuda.max_memory_reserved()}\n           if device == "cuda" else {})\n    return {"steps_timed": n_steps, "steps_per_epoch_10k": steps_epoch, "seconds": dt, "steps_per_s": n_steps / dt,\n            "params": int(sum(p.numel() for p in model.parameters())), "loss_first": losses[0], "loss_last": losses[-1],\n            "gpu_mem": mem, "n_adducts": len(adducts)}\n\n\ndef optional_stage(rep: dict, tag: str, fn):\n    """参考の段（fp16）。失敗してもエラーを記録して続ける。判定には使わない。"""\n    try:\n        rep["stages"][tag] = fn()\n    except StageTimeout:\n        raise\n    except Exception as e:  # noqa: BLE001\n        rep["stages"][tag] = {"error": f"{type(e).__name__}: {str(e)[:500]}", "reference_only": True}\n    return rep["stages"][tag]\n\n\ndef _env_info():\n    import torch\n    info = {"python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda,\n            "gpu_name": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,\n            "gpu_count": torch.cuda.device_count()}\n    try:\n        info["lscpu"] = subprocess.run(["lscpu"], capture_output=True, text=True, timeout=10).stdout[:3000]\n    except Exception as e:\n        info["lscpu"] = f"unavailable: {e}"\n    return info\n\n\ndef summarize(rep: dict) -> dict:\n    """判定は fp32 の外挿だけ。fp16 の参考の段は無い・失敗でも書ける。"""\n    S = rep["stages"]\n\n    def ref(tag, key):\n        v = S.get(tag)\n        if v is None:\n            return "not_run"\n        return v.get(key, v.get("error"))\n    return {\n        "gpu_name": rep.get("env", {}).get("gpu_name"),\n        "forward_sps_fp32": {p: S[f"embed_p{p}_fp32"]["forward_sps"] for p in PEAKS},\n        "forward_sps_fp16_reference": {p: ref(f"embed_p{p}_fp16", "forward_sps") for p in PEAKS},\n        "mlp_steps_per_s_fp32": S["mlp_fp32"]["steps_per_s"],\n        "mlp_steps_per_s_fp16_reference": ref("mlp_fp16", "steps_per_s"),\n        "gpu_mem_peak_max_allocated": max([v.get("gpu_mem", {}).get("max_allocated", 0) for v in S.values()\n                                           if isinstance(v, dict)] + [0]),\n        "gpu_mem_peak_max_reserved": max([v.get("gpu_mem", {}).get("max_reserved", 0) for v in S.values()\n                                          if isinstance(v, dict)] + [0]),\n        "ram_peak": ram_peak(),\n        "T_total_hours": rep["extrapolation"]["T_total_hours"],\n        "decision": rep["extrapolation"]["decision"],\n    }\n\n\ndef run(out_dir="/kaggle/working", work_dir="/kaggle/tmp_align", input_root="/kaggle/input", t0=None,\n        ckpt_override=None, dreams_root_override=None, n_spectra=N_SPECTRA, mlp_steps=None, bank_size=FP_BANK,\n        device=None) -> dict:\n    """全段を順に回し、段ごとに report を書き直す。どこで止まっても status と止まった段を残す。"""\n    t0 = time.time() if t0 is None else t0\n    deadline = t0 + STAGE_CUTOFF_S\n    out_dir, work_dir = Path(out_dir), Path(work_dir)\n    out_dir.mkdir(parents=True, exist_ok=True)\n    work_dir.mkdir(parents=True, exist_ok=True)\n    rpath = out_dir / "align_speed_report.json"\n    rep = {"issue": 99, "plan": "2026-11-14 noon item 5", "status": "running", "stage": None, "stages": {},\n           "prereg": "https://github.com/taichiiiiiiii/molecule-id-from-mass-spectra/issues/99#issuecomment-6870838727",\n           "constants": {"N_SPECTRA": n_spectra, "EMB_BATCH": EMB_BATCH, "PEAKS": list(PEAKS), "MLP_BATCH": MLP_BATCH,\n                         "MLP_K_NEG": MLP_K_NEG, "MLP_STEPS_PROD": MLP_STEPS_PROD, "FP_BITS": FP_BITS,\n                         "BUDGET_S": BUDGET_S, "STAGE_CUTOFF_S": STAGE_CUTOFF_S, "DREAMS_COMMIT": DREAMS_COMMIT,\n                         "CKPT_MD5": CKPT_MD5, "CKPT_SIZE": CKPT_SIZE},\n           "not_for_evaluation": "speed only; outputs are not used for v3 evaluation"}\n\n    def save():\n        rep["elapsed_s"] = _now(t0)\n        rep["ram_peak"] = ram_peak()\n        rpath.write_text(json.dumps(rep, indent=1, default=str))\n\n    def stage(name):\n        if time.time() > deadline:\n            raise StageTimeout(f"not starting stage {name}: elapsed {_now(t0)}s > {STAGE_CUTOFF_S}s")\n        rep["stage"] = name\n        save()\n\n    try:\n        import torch\n        dev = device or ("cuda" if torch.cuda.is_available() else "cpu")\n        stage("env")\n        rep["env"] = _env_info()\n        rep["device"] = dev\n        save()\n\n        stage("download_ckpt")\n        if ckpt_override is None:\n            ck = work_dir / "ssl_model.ckpt"\n            rep["stages"]["download_ckpt"] = download_ckpt(ck)\n            rep["md5_ok"] = True\n        else:\n            ck = ckpt_override\n            rep["stages"]["download_ckpt"] = {"override": True, "seconds": 0.0}\n        save()\n\n        stage("dreams_code")\n        if dreams_root_override is None:\n            rep["stages"]["dreams_code"] = fetch_dreams_code(work_dir)\n            droot = rep["stages"]["dreams_code"]["root"]\n        else:\n            droot = dreams_root_override\n            rep["stages"]["dreams_code"] = {"override": True, "root": droot, "seconds": 0.0}\n        save()\n\n        stage("load_model")\n        tl = time.time()\n        model, SP, DFA, linfo = load_dreams(droot, ck, dev)\n        linfo["seconds"] = round(time.time() - tl, 3)\n        if dev == "cuda":\n            linfo["gpu_mem_after_load"] = torch.cuda.memory_allocated()\n        rep["stages"]["load_model"] = linfo\n        save()\n        if ckpt_override is None:\n            try:\n                Path(ck).unlink()\n            except Exception:\n                pass\n\n        stage("select_spectra")\n        if input_root is None:\n            raise RuntimeError("input_root is None")\n        train_p, split_p = find_inputs(Path(input_root))\n        rep["inputs"] = {"train": str(train_p), "split": str(split_p)}\n        if train_p is None:\n            raise FileNotFoundError("competition train.parquet not found")\n        specs, sinfo = select_spectra(train_p, split_p, n_spectra)\n        rep["stages"]["select_spectra"] = sinfo\n        save()\n\n        # ---- fp32（判定に使う）\n        emb_keep, arrs = None, {}\n        for n_peaks in PEAKS:\n            stage(f"preprocess_p{n_peaks}")\n            arr, t_pre = preprocess(specs, SP, DFA, n_peaks)\n            arrs[n_peaks] = (arr, t_pre)\n            rep["stages"][f"preprocess_p{n_peaks}"] = {"seconds": t_pre, "sps": len(arr) / t_pre, "shape": list(arr.shape)}\n            save()\n            tag = f"embed_p{n_peaks}_fp32"\n            stage(tag)\n            emb, einfo = embed_speed(model, arr, dev, False, deadline=deadline)\n            einfo["sps_preprocess_plus_forward"] = len(arr) / (t_pre + einfo["seconds_forward"])\n            rep["stages"][tag] = einfo\n            save()\n            if n_peaks == PEAKS[0]:\n                emb_keep = emb\n        stage("mlp_fp32")\n        rep["stages"]["mlp_fp32"] = mlp_speed(emb_keep, specs, dev, False, n_steps=mlp_steps, bank_size=bank_size,\n                                              deadline=deadline)\n        save()\n        if dev == "cuda":\n            torch.cuda.empty_cache()\n\n        stage("extrapolate")\n        S = rep["stages"]\n        sps = {p: S[f"embed_p{p}_fp32"]["sps_preprocess_plus_forward"] for p in PEAKS}\n        fixed = S["download_ckpt"]["seconds"] + S["dreams_code"]["seconds"] + S["load_model"]["seconds"]\n        rep["extrapolation"] = extrapolate(sps, S["mlp_fp32"]["steps_per_s"], S["select_spectra"]["seconds"],\n                                           S["select_spectra"]["rows_read"], fixed)\n        rep["status"] = "ok_fp32"\n        save()\n\n        # ---- fp16 autocast（参考。失敗しても続ける。判定に使わない）\n        for n_peaks in PEAKS:\n            tag = f"embed_p{n_peaks}_fp16"\n            stage(tag)\n            arr, t_pre = arrs[n_peaks]\n\n            def _e(arr=arr, t_pre=t_pre):\n                _, info = embed_speed(model, arr, dev, True, deadline=deadline)\n                info["sps_preprocess_plus_forward"] = len(arr) / (t_pre + info["seconds_forward"])\n                info["reference_only"] = True\n                return info\n            optional_stage(rep, tag, _e)\n            save()\n        del arrs, model\n        if dev == "cuda":\n            torch.cuda.empty_cache()\n        stage("mlp_fp16")\n        optional_stage(rep, "mlp_fp16", lambda: dict(mlp_speed(emb_keep, specs, dev, True, n_steps=mlp_steps,\n                                                              bank_size=bank_size, deadline=deadline),\n                                                     reference_only=True))\n        save()\n\n        rep["summary"] = summarize(rep)\n        rep["status"] = "ok"\n        rep["stage"] = "done"\n    except Md5Mismatch as e:\n        rep["status"] = "md5_mismatch"\n        rep["error"] = str(e)\n        rep["md5_ok"] = False\n    except StageTimeout as e:\n        rep["status"] = "ok_fp32_reference_timeout" if "extrapolation" in rep else "timeout"\n        rep["error"] = str(e)\n    except Exception as e:  # noqa: BLE001\n        import traceback\n        rep["status"] = "ok_fp32_reference_error" if "extrapolation" in rep else "error"\n        rep["error"] = f"{type(e).__name__}: {e}"\n        rep["traceback"] = traceback.format_exc()[-4000:]\n    if "extrapolation" in rep and "summary" not in rep:\n        rep["summary"] = summarize(rep)\n    save()\n    return rep\n\n\nif __name__ == "__main__":\n    r = run()\n    print(json.dumps({k: r.get(k) for k in ("status", "stage", "error", "summary", "elapsed_s")}, indent=1, default=str))\n'

In [ ]:
import json, sys, time, shutil
from pathlib import Path
T0 = time.time()
WORK = Path('/kaggle/tmp_align'); WORK.mkdir(parents=True, exist_ok=True)
(WORK / 'align_speed_trial.py').write_text(ALIGN_SRC)
sys.path.insert(0, str(WORK))
import align_speed_trial as A
assert A.CKPT_MD5 == '4deee91dd4486c2b6cbcde1f200b8f50' and A.DREAMS_COMMIT.startswith('dbec3a0')
rep = A.run(out_dir='/kaggle/working', work_dir=str(WORK), input_root='/kaggle/input', t0=T0)
print(json.dumps({k: rep.get(k) for k in ('status', 'stage', 'error', 'summary', 'elapsed_s')}, indent=1, default=str))
shutil.rmtree(WORK, ignore_errors=True)
print('OUTPUTS', sorted(p.name for p in Path('/kaggle/working').iterdir()))